# CFFair

In the causal graph of our generated data, all observable features ($X$) are causally downstream of the sensitive attribute ($S$). Level 2 fairness requires using only observable features that are completely unaffected by $S$. For our datasets, that leaves us with zero features to use, making a Level 2 model essentially a random guesser.

Level 3 fairness solves this by digging deeper. It bypasses the tainted observable world entirely and predicts the target ($Y$) using only the innate, unobserved background variables ($U$). Because $U$ is mathematically independent of $S$ at birth (in a causal sense), any model that relies exclusively on $U$ is guaranteed to be 100% counterfactually fair.

Our Approach to Replicate the ProcessorThe original researchers had to use heavy Bayesian math (PyStan) to "guess" $U$ because it's invisible in real-world data (like the Law School dataset).Because we generated our own data, we already know $U$. Our approach is to act as the perfect "Oracle." We will strictly drop all $X$ and $S$ columns during training. We will build a fast Neural Network (replicating their "Emulator") that learns to predict $Y$ using only your ground-truth $U$ columns. This gives the theoretical maximum accuracy a perfectly fair model can achieve on our data

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
warnings.filterwarnings('ignore')

"""
====================================================================
COUNTERFACTUAL FAIRNESS (CFFAIR) - LEVEL 3 ORACLE
====================================================================
APPROACH JUSTIFICATION:
To achieve Level 3 Counterfactual Fairness (Kusner et al., 2017), a 
model must base its predictions strictly on non-descendants of the 
sensitive attribute (S). In our structural causal model, the only 
true non-descendants are the latent background variables (U).

Since our datasets contain the ground-truth generated U variables 
(e.g., U1, U_behav), we completely bypass the need for PyStan/MCMC 
inference. We act as a perfect "Oracle Emulator" by dropping all 
observable features (X) and training exclusively on (U) to predict (Y).
====================================================================
"""

# ==========================================
# STEP 1: CFFAIR ORACLE ARCHITECTURE
# ==========================================
class CFFair_Oracle(nn.Module):
    """ 
    The Level 3 Predictor (Emulator).
    Notice that the input dimension is strictly u_dim (the number of U columns).
    It never sees X or S during its forward pass.
    """
    def __init__(self, u_dim, hidden_dim=64):
        super(CFFair_Oracle, self).__init__()
        # Standard Multi-Layer Perceptron (MLP)
        self.fc1 = nn.Linear(u_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u):
        h = F.relu(self.fc1(u))
        h = F.relu(self.fc2(h))
        # Sigmoid output for binary classification (predicting Y)
        return torch.sigmoid(self.out(h))


# ==========================================
# STEP 2: TRAINING & PREDICTION FUNCTIONS
# ==========================================
def train_cffair_oracle(model, U_tensor, Y_tensor, epochs=100, batch_size=128, lr=1e-3):
    """
    Trains the Oracle model to predict Y using only U.
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Dataset consists ONLY of U and Y. X and S are strictly excluded.
    dataset = TensorDataset(U_tensor, Y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        for batch_u, batch_y in dataloader:
            optimizer.zero_grad()
            
            # Predict Y from U
            y_pred = model(batch_u)
            
            # Standard Binary Cross Entropy Loss
            loss = F.binary_cross_entropy(y_pred, batch_y)
            
            loss.backward()
            optimizer.step()
            
    return model


def predict_cffair(model, U_tensor):
    """
    Generates final Counterfactually Fair predictions using only U.
    """
    model.eval()
    with torch.no_grad():
        probs = model(U_tensor)
        preds = (probs >= 0.5).float()
        return probs.numpy().flatten(), preds.numpy().flatten()

# Synthetic data processing

In [4]:
# ==========================================
# SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = r"C:\Users\patri\Documents\Master Semesters\2nd semester\APA\model_testing\SF_and_Hidden_Nodes_Data_Simulation\results\generated_160_csv_10_seeds\full_data" #r"C:\Users\patri\Documents\Master Semesters\2nd semester\APA\model_testing\FairPFN\synthetic_data"
output_file = "CFFair_syn_results.csv"

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with CFFair Oracle...")
    
    try:
        df = pd.read_csv(file_path)
        
        # --- DYNAMIC FEATURE DISCOVERY & CAUSAL ISOLATION ---
        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith(('U', 'H', 'C'))]
        
        if 'Y' not in df.columns or len(s_cols) == 0 or len(u_cols) == 0:
            print(f"  [SKIPPED] Missing 'Y', 'S', or 'U' columns. CFFair requires unobserved variables (U).")
            continue
            
        # Target variable
        y_data = df['Y'].values.reshape(-1, 1)
        
        # Sensitive attribute (Kept ONLY for evaluation, never for training)
        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        S_data = df[[s_target]].values
        
        # Unobserved Innate Traits (This is the ONLY data the model will see)
        U_features = df[u_cols].values
        
        # Enforce the FairPFN size limit for fair comparative benchmarking
        train_size = min(1000, int(len(U_features) * 0.8))
        
        # Split Data (Notice X is completely missing from this split!)
        U_train, U_test, y_train, y_test, S_train, S_test = train_test_split(
            U_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )
        
        # Convert to Tensors
        U_train_t = torch.tensor(U_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        U_test_t  = torch.tensor(U_test, dtype=torch.float32)
        
        u_dim = U_train.shape[1]
        
        # --- MODEL INFERENCE (THE EMULATOR) ---
        oracle_model = CFFair_Oracle(u_dim=u_dim)
        oracle_model = train_cffair_oracle(oracle_model, U_train_t, y_train_t, epochs=100)
        
        # --- EVALUATION ---
        prob_preds, predictions = predict_cffair(oracle_model, U_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()
            
        # 1. Prediction Metrics (How accurate are we using ONLY innate traits?)
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)
        
        # 2. Fairness Metrics (Did dropping X actually make the predictions fair?)
        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)
        
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')
        
        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        
        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        
        equal_opp_diff = abs(tpr_1 - tpr_0)
        
        # --- RECORD DATA ---
        master_results.append({
            "model_name": "CFFair_Oracle",
            "name_dataset": dataset_name,
            "n_S": len(s_cols),
            "n_X": len(x_cols),  # We log this just to record the shape of the original file
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })
        
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")
        
    except Exception as e:
        print(f"  [ERROR] Failed to process. Reason: {str(e)}")

# ==========================================
# 3. SAVE AGGREGATED RESULTS
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets successfully.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")

Found 80 synthetic datasets to process.

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_0_8_nodes_full_data.csv with CFFair Oracle...
  [SKIPPED] Missing 'Y', 'S', or 'U' columns. CFFair requires unobserved variables (U).

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_1_8_nodes_full_data.csv with CFFair Oracle...
  [SKIPPED] Missing 'Y', 'S', or 'U' columns. CFFair requires unobserved variables (U).

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_2_8_nodes_full_data.csv with CFFair Oracle...
  [SKIPPED] Missing 'Y', 'S', or 'U' columns. CFFair requires unobserved variables (U).

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_3_8_nodes_full_data.csv with CFFair Oracle...
  [SKIPPED] Missing 'Y', 'S', or 'U' columns. CFFair requires unobserved variables (U).

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_4_8_nodes

# Semi-synthetic data processing

In [2]:
# ==========================================
# SEMI-SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = r"C:\Users\patri\Documents\Master Semesters\2nd semester\APA\model_testing\FairPFN\semi_synthetic_data"
output_file = "CFFair_semi_syn_results.csv"

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} semi-synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with CFFair Oracle...")
    
    try:
        df = pd.read_csv(file_path)
        
        # --- FILENAME PARAMETER EXTRACTION ---
        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        # --- DYNAMIC FEATURE DISCOVERY & CAUSAL ISOLATION ---
        # Target variable
        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values.reshape(-1, 1)
        
        # Unobserved Innate Traits (The ONLY input for the model)
        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]
        if not u_cols:
            print("  [SKIPPED] Missing Unobserved Variables (U_/H_/C_). CFFair requires these to emulate fairness.")
            continue
        U_features = df[u_cols].values
        
        # Observable Features (Only recorded for the schema metrics, dropped from training)
        x_cols = [col for col in df.columns if col.startswith('X_')]
        
        # Protected Attribute (Used ONLY for evaluation, never for training)
        s_cols = [col for col in df.columns if col.startswith('S') or col.startswith('G')]
        has_protected = len(s_cols) > 0
        
        if has_protected:
            s_target = s_cols[0]
            S_data = df[[s_target]].values
        else:
            print("  [INFO] No protected column found. Evaluating standard predictions without fairness metrics.")
            S_data = np.zeros((len(df), 1))
        
        # Enforce the FairPFN size limit
        train_size = min(1000, int(len(U_features) * 0.8))
        
        # Split Data (Notice X is completely missing from this split!)
        U_train, U_test, y_train, y_test, S_train, S_test = train_test_split(
            U_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )
        
        # Convert to Tensors
        U_train_t = torch.tensor(U_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        U_test_t  = torch.tensor(U_test, dtype=torch.float32)
        
        u_dim = U_train.shape[1]
        
        # --- MODEL INFERENCE (THE EMULATOR) ---
        oracle_model = CFFair_Oracle(u_dim=u_dim)
        oracle_model = train_cffair_oracle(oracle_model, U_train_t, y_train_t, epochs=100)
        
        # --- EVALUATION ---
        prob_preds, predictions = predict_cffair(oracle_model, U_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()
            
        # 1. Prediction Metrics
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)
        
        # 2. Fairness Metrics
        if has_protected:
            group_1_mask = (S_test_flat == 1)
            group_0_mask = (S_test_flat == 0)
            
            rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
            rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
            
            stat_parity_diff = abs(rate_1 - rate_0)
            disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')
            
            y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
            tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
            
            y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
            tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
            
            equal_opp_diff = abs(tpr_1 - tpr_0)
        else:
            rate_1 = rate_0 = stat_parity_diff = disp_impact = equal_opp_diff = np.nan
        
        # --- RECORD DATA ---
        master_results.append({
            "model_name": "CFFair_Oracle",
            "name_dataset": dataset_name,
            "bias_level": bias_level,
            "threshold": threshold,
            "data_type": data_type,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4) if pd.notna(stat_parity_diff) else np.nan,
            "Disparate_Impact_Ratio": round(disp_impact, 4) if pd.notna(disp_impact) else np.nan,
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4) if pd.notna(equal_opp_diff) else np.nan,
            "Pos_Rate_S1": round(rate_1, 4) if pd.notna(rate_1) else np.nan,
            "Pos_Rate_S0": round(rate_0, 4) if pd.notna(rate_0) else np.nan
        })
        
        if has_protected:
            print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")
        else:
            print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: N/A")
        
    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

# ==========================================
# 3. SAVE AGGREGATED RESULTS
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets successfully.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")

Found 800 semi-synthetic datasets to process.

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_BIASED_ALL.csv with CFFair Oracle...
  [SUCCESS] AUC: 0.957 | ATE: 0.073

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_BIASED_FAIRNESS.csv with CFFair Oracle...
  [SKIPPED] Missing Unobserved Variables (U_/H_/C_). CFFair requires these to emulate fairness.

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_BIASED_MASKED.csv with CFFair Oracle...
  [SKIPPED] Missing Unobserved Variables (U_/H_/C_). CFFair requires these to emulate fairness.

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_FAIR_ALL.csv with CFFair Oracle...
  [SUCCESS] AUC: 0.801 | ATE: 0.017

Processing: HR_N1000_HighBias_Thresh1.0_Seed100_BIASED_ALL.csv with CFFair Oracle...
  [SUCCESS] AUC: 0.950 | ATE: 0.135

Processing: HR_N1000_HighBias_Thresh1.0_Seed100_BIASED_FAIRNESS.csv with CFFair Oracle...
  [SKIPPED] Missing Unobserved Variables (U_/H_/C_). CFFair requires these to emulate fairness.

Processing: HR_N1000_HighB